In [3]:
# !pip install -q groq

# from google.colab import userdata
# from groq import Groq

# # Retrieve key from Colab Secrets
# groq_api_key = userdata.get("GROQ_API_KEY")
# client = Groq(api_key=groq_api_key)

# # 1. Print all available models on your account
# models = client.models.list()
# available_model_ids = [m.id for m in models.data]
# print("Available models in your account:", available_model_ids)

# # 2. Test completion with standard fast model
# active_model = "llama-3.1-8b-instant" if "llama-3.1-8b-instant" in available_model_ids else available_model_ids[0]
# print(f"\nTesting with model: {active_model}...")

# completion = client.chat.completions.create(
#     model=active_model,
#     messages=[
#         {"role": "user", "content": "Hello! Reply with 'Groq is ready' if you can hear me."}
#     ],
#     temperature=0.1,
#     max_tokens=20
# )

# print("\nResponse from Groq:", completion.choices[0].message.content)

Available models in your account: ['meta-llama/llama-prompt-guard-2-86m', 'qwen/qwen3.6-27b', 'whisper-large-v3', 'groq/compound-mini', 'qwen/qwen3.8-27b', 'groq/compound', 'canopylabs/orpheus-v1-english', 'canopylabs/orpheus-arabic-saudi', 'allam-2-7b', 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'meta-llama/llama-prompt-guard-2-22m', 'whisper-large-v3-turbo', 'openai/gpt-oss-safeguard-20b']

Testing with model: meta-llama/llama-prompt-guard-2-86m...

Response from Groq: 0.0009654992609284818


In [4]:
!pip install -q qdrant-client sentence-transformers groq datasets pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 29.5 MB/s eta 0:00:00


In [6]:
from google.colab import userdata
from groq import Groq

# 1. Load API Key from Colab Secrets
groq_api_key = userdata.get("GROQ_API_KEY")
client_groq = Groq(api_key=groq_api_key)

# 2. Automatically select an active chat generation model (filtering out guard models)
available_models = [m.id for m in client_groq.models.list().data]
chat_models = [m for m in available_models if "guard" not in m and "whisper" not in m]

# Prefer Llama 3.1 / 3.3 or pick the first available chat model
PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant", "llama3-70b-8192", "mixtral-8x7b-32768"]
LLM_MODEL = next((m for m in PREFERRED if m in chat_models), chat_models[0])

print(f"Selected LLM for RAG Generation: [{LLM_MODEL}]")

Selected LLM for RAG Generation: [groq/compound]


In [7]:
import uuid
import pandas as pd
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# 1. Load Bitext Customer Support Dataset
print("Loading Bitext Knowledge Base Dataset...")
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = pd.DataFrame(dataset["train"])

# Drop exact duplicate pairs to keep retrieval fast
df = df.drop_duplicates(subset=["instruction", "response"]).reset_index(drop=True)
print(f"Total unique knowledge base pairs to index: {len(df)}")

# 2. Load Embedding Model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading embedding model '{EMBEDDING_MODEL_NAME}'...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
EMBEDDING_DIM = 384

# 3. Initialize Local Persistent Qdrant Client
COLLECTION_NAME = "customer_support_kb"
qdrant = QdrantClient(path="./qdrant_db")

# Recreate collection if exists
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE)
)

# 4. Batch Embed & Ingest into Qdrant
BATCH_SIZE = 256
print(f"Indexing {len(df)} support chunks into Qdrant...")

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch_df = df.iloc[i : i + BATCH_SIZE]
    embeddings = embedder.encode(
        batch_df["instruction"].tolist(),
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True
    )

    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector=embeddings[idx].tolist(),
            payload={
                "instruction": row["instruction"],
                "response": row["response"],
                "intent": row["intent"],
                "category": row["category"]
            }
        )
        for idx, (_, row) in enumerate(batch_df.iterrows())
    ]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"✅ Ingestion complete! Total indexed points: {qdrant.count(COLLECTION_NAME).count}")

Loading Bitext Knowledge Base Dataset...


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexing 26872 support chunks into Qdrant...


  0%|          | 0/105 [00:00<?, ?it/s]

/tmp/ipykernel_1477/1838163135.py:63: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20224 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant.upsert(collection_name=COLLECTION_NAME, points=points)


✅ Ingestion complete! Total indexed points: 26872


In [11]:
def retrieve_support_chunks(query, top_k=3, score_threshold=None):
    """
    Retrieves top-k matching support responses from Qdrant.
    """
    query_vector = embedder.encode(query, convert_to_numpy=True).tolist()

    response = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        score_threshold=score_threshold,
    )

    results = response.points
    chunks = []
    for res in results:
        chunks.append({
            "score": round(res.score, 4),
            "instruction": res.payload.get("instruction"),
            "response": res.payload.get("response"),
            "intent": res.payload.get("intent"),
            "category": res.payload.get("category")
        })
    return chunks
def generate_grounded_answer(user_message: str, detected_sentiment: str, retrieved_chunks: list) -> str:
    """
    Queries Groq using the retrieved context and tone modulation.
    """
    # Fallback if no relevant knowledge base entry is found
    if not retrieved_chunks:
        if detected_sentiment == "frustrated":
            return (
                "I understand your frustration, and I apologize for the difficulty. "
                "Our knowledge base does not cover your specific request. "
                "I am escalating your inquiry to a human agent right away."
            )
        return (
            "I'm sorry, but our support documentation does not cover that question. "
            "Would you like me to connect you with a live human support specialist?"
        )

    # Format context chunks
    context_text = ""
    for idx, chunk in enumerate(retrieved_chunks, 1):
        context_text += f"Support Response {idx} [{chunk['category']} / {chunk['intent']}]:\n{chunk['response']}\n\n"

    # Prompt Template matching project specification
    system_prompt = (
        "You are a helpful, professional customer support assistant for an online retailer. "
        "Answer the customer's question using ONLY the information in the retrieved support responses below. "
        f"If the customer sounds frustrated ({detected_sentiment}), acknowledge that before answering. "
        "If the retrieved context does not cover the question, say so honestly and offer to escalate to a human "
        "agent rather than guessing."
    )

    user_prompt = (
        f"Context (retrieved past support responses):\n"
        f"{context_text.strip()}\n\n"
        f"Customer question: \"{user_message}\""
    )

    completion = client_groq.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=512
    )

    return completion.choices[0].message.content

In [12]:
test_cases = [
    {
        "query": "How can I track the delivery status of my package?",
        "sentiment": "neutral"
    },
    {
        "query": "I asked for a refund two weeks ago and got nothing! This is ridiculous!",
        "sentiment": "frustrated"
    },
    {
        "query": "Can I update my shipping address after placing an order?",
        "sentiment": "neutral"
    },
    {
        "query": "Who won the FIFA World Cup in 1998?",  # Out of domain query
        "sentiment": "neutral"
    }
]

for tc in test_cases:
    query = tc["query"]
    sentiment = tc["sentiment"]

    print("=" * 80)
    print(f"Customer Query: \"{query}\" (Detected Sentiment: [{sentiment}])")

    # 1. Retrieve
    retrieved = retrieve_support_chunks(query, top_k=3)
    print(f"Retrieved Chunks: {len(retrieved)}")

    # 2. Generate
    response = generate_grounded_answer(query, sentiment, retrieved)
    print(f"\nAssistant Response:\n{response}\n")

Customer Query: "How can I track the delivery status of my package?" (Detected Sentiment: [neutral])
Retrieved Chunks: 3

Assistant Response:
**How to track the delivery status of your package**

Based on the three support responses we have, here’s the complete reasoning and the steps you can follow:

1. **Use the online tracking system**  
   - Go to our website and find the **‘Order Status’** or **‘Track Order’** page.  
   - Enter either your **Order Number** or **Tracking Number**.  
   - The system will show the most up‑to‑date location of your package and its estimated delivery date.

2. **Log in to your account**  
   - Sign in on our website.  
   - Navigate to **‘Order History’** or **‘Track Your Order’**.  
   - You’ll see detailed information about the current status and expected arrival.

3. **Track with the tracking number (no account needed)**  
   - Locate the tracking number in the shipping‑confirmation email you received.  
   - Visit the website’s **tracking page**, e

In [13]:
!zip -r qdrant_db.zip ./qdrant_db
from google.colab import files
files.download("qdrant_db.zip")

  adding: qdrant_db/ (stored 0%)
  adding: qdrant_db/meta.json (deflated 57%)
  adding: qdrant_db/collection/ (stored 0%)
  adding: qdrant_db/collection/customer_support_kb/ (stored 0%)
  adding: qdrant_db/collection/customer_support_kb/storage.sqlite (deflated 55%)
  adding: qdrant_db/.lock (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>